# vLLM Serving Benchmark on T4: Batching, Quantization, Tensor Parallel

**Question.** On commodity, older GPUs (NVIDIA Tesla T4, compute capability 7.5, no bf16), how much do the main
vLLM serving levers actually move throughput and latency for small instruction models?

1. **Continuous batching** — offline generated tokens/s as `max_num_seqs` grows (1 → 128).
2. **Online serving** — OpenAI-compatible server under streaming load: TTFT, inter-token latency, end-to-end latency, output tokens/s at concurrency 1–32.
3. **Weight quantization** — fp16 vs AWQ (4-bit) for the same base model: speed, memory, and a small exact-match quality check.
4. **Tensor parallelism** — TP=1 vs TP=2 on the two T4s.
5. **Prefix caching** — on vs off with a long shared system prompt.

Every number below is produced by this notebook run and saved to `/kaggle/working/results.json`. Nothing is hand-edited.

## Setup

- **Models (public, no token):** `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-0.5B-Instruct-AWQ`, `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct-AWQ`.
- **Engine:** vLLM pinned to a release whose V0 engine supports compute capability 7.5 (xformers attention). `dtype=half` because T4 has no bf16. `VLLM_USE_V1=0`. Those releases have no Python 3.13 wheels, so vLLM runs in an isolated uv-managed Python 3.12 venv; the notebook kernel only orchestrates, tabulates and plots.
- **Isolation:** each engine configuration runs in a **separate subprocess** (offline) or a separately launched server, and GPU memory is checked to be released before the next one.
- **Workload:** synthetic engineering-style prompts with a fixed seed, chat template applied, greedy decoding, `max_tokens=128` with `ignore_eos=True` so every request generates exactly the same number of tokens (throughput comparable across configs).
- **Online metrics:** streaming `/v1/chat/completions`. TTFT = time to first content chunk; ITL = gap between consecutive content chunks; TPOT = (e2e − TTFT)/(tokens − 1); output tokens/s = all generated tokens / wall time of the level. Requests per level = max(16, 4 × concurrency).
- **Quality check:** 50 deterministic prompts (25 two-digit additions, 25 "reply with this word in uppercase"), temperature 0, exact match. This is a sanity check for gross quantization damage, not a benchmark of model quality.

In [ ]:
import os, shutil, subprocess, sys, time
T_START = time.time()
print("notebook kernel python:", sys.version)

# The Kaggle image ships Python 3.13, but T4-compatible vLLM releases (V0 engine, xformers,
# torch 2.5.x) only publish wheels for Python <= 3.12. vLLM therefore runs in an isolated
# uv-managed Python 3.12 venv; every benchmark process uses VPY. The venv lives in /tmp so the
# multi-GB environment is not saved as notebook output.
VENV = "/tmp/vllm-venv"
VPY = f"{VENV}/bin/python"

def sh(args, tail=1500):
    r = subprocess.run(args, capture_output=True, text=True)
    print(" ".join(args[:6]), "... exit", r.returncode)
    if r.returncode != 0 or tail:
        print((r.stdout + r.stderr)[-tail:])
    return r

sh([sys.executable, "-m", "pip", "install", "-q", "uv"], tail=300)
sh([sys.executable, "-m", "uv", "venv", VENV, "--python", "3.12"], tail=500)

CANDIDATES = [("vllm==0.6.6.post1", "transformers==4.47.1"),
              ("vllm==0.7.3", "transformers==4.49.0"),
              ("vllm==0.6.4.post1", "transformers==4.46.3")]
INSTALLED = None
for vspec, tspec in CANDIDATES:
    sh([sys.executable, "-m", "uv", "pip", "install", "--python", VPY, vspec, tspec, "httpx"], tail=1500)
    chk = sh([VPY, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__, torch.version.cuda, "
              "torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],
             tail=1500)
    if chk.returncode == 0 and "True" in chk.stdout:
        INSTALLED = vspec
        break
assert INSTALLED, "no vLLM candidate imported successfully with CUDA available"
print("Using", INSTALLED, f"(setup took {time.time() - T_START:.0f}s)")

In [ ]:
import json
ENV_PROBE = r'''
import json, platform, torch, vllm, transformers
try:
    import xformers; xf = xformers.__version__
except Exception:
    xf = None
print(json.dumps({
    "python": platform.python_version(),
    "vllm": vllm.__version__, "torch": torch.__version__, "torch_cuda": torch.version.cuda,
    "transformers": transformers.__version__, "xformers": xf,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [{"name": torch.cuda.get_device_name(i),
              "capability": ".".join(map(str, torch.cuda.get_device_capability(i))),
              "total_mem_gib": round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2)}
             for i in range(torch.cuda.device_count())],
}))
'''
ENV = json.loads(subprocess.run([VPY, "-c", ENV_PROBE], capture_output=True, text=True).stdout)
ENV["benchmark_python"] = "isolated uv venv (see 'python')"
ENV["notebook_kernel_python"] = sys.version.split()[0]
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
ENV["nvidia_smi"] = smi.splitlines()
print(json.dumps(ENV, indent=2))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/offline_bench.py
"""Offline throughput / quality benchmark for one vLLM engine config.

Runs in its own process so GPU memory is fully released between configs.
Writes a single JSON object to --out.
"""
import argparse
import json
import os
import random
import re
import subprocess
import time

os.environ.setdefault("VLLM_USE_V1", "0")

TOPICS = [
    "cache invalidation", "backpressure in message queues", "B-tree indexing",
    "consistent hashing", "rate limiting with token buckets", "database connection pooling",
    "idempotent API design", "retry with exponential backoff", "memory fragmentation",
    "lock-free queues", "write-ahead logging", "vector clocks", "circuit breakers",
    "blue-green deployments", "garbage collection pauses", "TCP congestion control",
    "bloom filters", "leader election", "schema migrations", "load shedding",
    "CPU cache locality", "profiling a slow service", "observability with traces",
    "unit vs integration tests", "floating point rounding errors", "thread pools",
    "event sourcing", "sharding a relational database", "zero-downtime upgrades",
    "numerical stability in softmax", "mixed precision training", "KV caches in transformers",
]
CONTEXTS = [
    "a high-traffic web backend", "an embedded control system", "a data pipeline",
    "a mobile app", "a batch analytics job", "a real-time trading system",
    "a machine learning serving stack", "a distributed key-value store",
]


def make_prompts(n, seed=0):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        topic, ctx = rng.choice(TOPICS), rng.choice(CONTEXTS)
        k = rng.randint(3, 6)
        out.append(
            f"Explain {topic} in the context of {ctx}. "
            f"Give {k} concrete, practical points and one common pitfall."
        )
    return out


def long_system_prompt():
    rules = []
    for i in range(1, 31):
        t = TOPICS[i % len(TOPICS)]
        c = CONTEXTS[i % len(CONTEXTS)]
        rules.append(
            f"Rule {i}: when the question touches {t} in {c}, state assumptions first, "
            f"prefer measurable claims, and mention one failure mode."
        )
    return "You are a careful senior engineer reviewing design questions.\n" + "\n".join(rules)


def quality_set(seed=1234):
    rng = random.Random(seed)
    items = []
    for _ in range(25):
        a, b = rng.randint(10, 99), rng.randint(10, 99)
        items.append((f"What is {a} + {b}? Reply with only the number.", str(a + b), "int"))
    words = ["engine", "valve", "sensor", "gearbox", "turbine", "rotor", "filter", "bearing",
             "piston", "nozzle", "spring", "cable", "switch", "relay", "motor", "pump",
             "shaft", "clutch", "brake", "wheel", "flange", "gasket", "bolt", "frame", "hinge"]
    for w in words:
        items.append((f"Reply with only the word '{w}' in uppercase letters, nothing else.",
                      w.upper(), "word"))
    return items


def score(text, gold, kind):
    if kind == "int":
        m = re.search(r"-?\d+", text)
        return m is not None and m.group(0) == gold
    m = re.search(r"[A-Za-z]+", text)
    return m is not None and m.group(0) == gold


def nvidia_smi_mem():
    try:
        q = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        return [{"gpu": int(r.split(",")[0]), "used_mib": int(r.split(",")[1]),
                 "total_mib": int(r.split(",")[2])} for r in q]
    except Exception as e:  # noqa: BLE001
        return {"error": str(e)}


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--quantization", default=None)
    p.add_argument("--dtype", default="half")
    p.add_argument("--tp", type=int, default=1)
    p.add_argument("--max-num-seqs", type=int, default=128)
    p.add_argument("--num-prompts", type=int, default=128)
    p.add_argument("--max-tokens", type=int, default=128)
    p.add_argument("--gpu-mem", type=float, default=0.85)
    p.add_argument("--max-model-len", type=int, default=2048)
    p.add_argument("--prefix-caching", action="store_true")
    p.add_argument("--shared-prefix", action="store_true")
    p.add_argument("--quality", action="store_true")
    p.add_argument("--out", required=True)
    a = p.parse_args()

    import torch
    from vllm import LLM, SamplingParams

    t0 = time.perf_counter()
    llm = LLM(
        model=a.model, dtype=a.dtype, quantization=a.quantization,
        tensor_parallel_size=a.tp, max_num_seqs=a.max_num_seqs,
        gpu_memory_utilization=a.gpu_mem, max_model_len=a.max_model_len,
        enable_prefix_caching=a.prefix_caching, disable_custom_all_reduce=True, seed=0,
    )
    load_s = time.perf_counter() - t0
    tok = llm.get_tokenizer()

    def fmt(user, system=None):
        msgs = ([{"role": "system", "content": system}] if system else []) + [
            {"role": "user", "content": user}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    system = long_system_prompt() if a.shared_prefix else None
    prompts = [fmt(u, system) for u in make_prompts(a.num_prompts)]

    llm.generate([fmt("Say hello.")], SamplingParams(temperature=0, max_tokens=8), use_tqdm=False)

    sp = SamplingParams(temperature=0, max_tokens=a.max_tokens, ignore_eos=True)
    t1 = time.perf_counter()
    outs = llm.generate(prompts, sp, use_tqdm=False)
    elapsed = time.perf_counter() - t1
    gen_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    prompt_tokens = sum(len(o.prompt_token_ids) for o in outs)

    res = {
        "config": vars(a),
        "load_seconds": round(load_s, 2),
        "num_requests": len(prompts),
        "prompt_tokens": prompt_tokens,
        "generated_tokens": gen_tokens,
        "elapsed_seconds": round(elapsed, 3),
        "generated_tokens_per_s": round(gen_tokens / elapsed, 1),
        "total_tokens_per_s": round((gen_tokens + prompt_tokens) / elapsed, 1),
        "requests_per_s": round(len(prompts) / elapsed, 3),
        "num_gpu_kv_blocks": getattr(llm.llm_engine.cache_config, "num_gpu_blocks", None),
        "torch_max_memory_allocated_gib_gpu0": round(torch.cuda.max_memory_allocated() / 2**30, 3),
        "nvidia_smi_after_run": nvidia_smi_mem(),
    }

    if a.quality:
        items = quality_set()
        qouts = llm.generate([fmt(q) for q, _, _ in items],
                             SamplingParams(temperature=0, max_tokens=16), use_tqdm=False)
        details = []
        for (q, gold, kind), o in zip(items, qouts):
            text = o.outputs[0].text.strip()
            details.append({"prompt": q, "gold": gold, "output": text[:60],
                            "correct": score(text, gold, kind)})
        res["quality"] = {
            "n": len(details),
            "exact_match": round(sum(d["correct"] for d in details) / len(details), 3),
            "arith_exact_match": round(sum(d["correct"] for d in details[:25]) / 25, 3),
            "format_exact_match": round(sum(d["correct"] for d in details[25:]) / 25, 3),
            "details": details,
        }

    with open(a.out, "w") as f:
        json.dump(res, f, indent=2)
    print(json.dumps({k: v for k, v in res.items() if k != "quality"}))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/online_client.py
"""Streaming load generator for a running vLLM OpenAI-compatible server.

Measures per-request TTFT, inter-token latency (ITL), time per output token
(TPOT) and end-to-end latency at several concurrency levels.
"""
import argparse
import asyncio
import json
import statistics
import time

import httpx

from offline_bench import make_prompts


def pct(xs, q):
    if not xs:
        return None
    xs = sorted(xs)
    k = (len(xs) - 1) * q
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return round(xs[lo] + (xs[hi] - xs[lo]) * (k - lo), 2)


async def one_request(client, url, model, prompt, max_tokens):
    payload = {
        "model": model, "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": 0, "stream": True,
        "ignore_eos": True, "stream_options": {"include_usage": True},
    }
    t0 = time.perf_counter()
    first = last = None
    itls, chunks, usage_tokens = [], 0, None
    async with client.stream("POST", url, json=payload) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            if not line.startswith("data: "):
                continue
            data = line[6:]
            if data.strip() == "[DONE]":
                break
            j = json.loads(data)
            if j.get("usage"):
                usage_tokens = j["usage"].get("completion_tokens")
            ch = j.get("choices") or []
            if ch and ch[0].get("delta", {}).get("content"):
                now = time.perf_counter()
                if first is None:
                    first = now
                else:
                    itls.append(now - last)
                last = now
                chunks += 1
    end = time.perf_counter()
    tokens = usage_tokens or chunks
    return {"ttft": first - t0, "e2e": end - t0, "itls": itls, "tokens": tokens,
            "tpot": (end - first) / max(tokens - 1, 1)}


async def run_level(url, model, concurrency, n_requests, max_tokens):
    prompts = make_prompts(n_requests, seed=concurrency)
    sem = asyncio.Semaphore(concurrency)
    limits = httpx.Limits(max_connections=concurrency + 4, max_keepalive_connections=concurrency + 4)
    async with httpx.AsyncClient(timeout=None, limits=limits) as client:
        await one_request(client, url, model, "Say hello.", 8)

        async def guarded(pr):
            async with sem:
                try:
                    return await one_request(client, url, model, pr, max_tokens)
                except Exception as e:  # noqa: BLE001
                    return {"error": repr(e)}

        t0 = time.perf_counter()
        rs = await asyncio.gather(*(guarded(pr) for pr in prompts))
        wall = time.perf_counter() - t0

    ok = [r for r in rs if "error" not in r]
    ms = lambda xs: [round(x * 1000, 2) for x in xs]  # noqa: E731
    ttft, e2e, tpot = ms([r["ttft"] for r in ok]), ms([r["e2e"] for r in ok]), ms([r["tpot"] for r in ok])
    itl = ms([x for r in ok for x in r["itls"]])
    out_tokens = sum(r["tokens"] for r in ok)
    return {
        "concurrency": concurrency, "num_requests": n_requests, "errors": len(rs) - len(ok),
        "wall_seconds": round(wall, 3), "output_tokens": out_tokens,
        "output_tokens_per_s": round(out_tokens / wall, 1),
        "requests_per_s": round(len(ok) / wall, 3),
        "ttft_ms": {"p50": pct(ttft, .5), "p95": pct(ttft, .95), "mean": round(statistics.mean(ttft), 2) if ttft else None},
        "itl_ms": {"p50": pct(itl, .5), "p95": pct(itl, .95)},
        "tpot_ms": {"p50": pct(tpot, .5), "p95": pct(tpot, .95)},
        "e2e_ms": {"p50": pct(e2e, .5), "p95": pct(e2e, .95)},
        "first_error": next((r["error"] for r in rs if "error" in r), None),
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--levels", default="1,4,16,32")
    p.add_argument("--max-tokens", type=int, default=128)
    p.add_argument("--out", required=True)
    a = p.parse_args()
    url = f"http://127.0.0.1:{a.port}/v1/chat/completions"
    results = []
    for c in [int(x) for x in a.levels.split(",")]:
        r = asyncio.run(run_level(url, a.model, c, max(16, 4 * c), a.max_tokens))
        print(json.dumps({k: r[k] for k in ("concurrency", "output_tokens_per_s", "ttft_ms", "itl_ms", "errors")}))
        results.append(r)
    with open(a.out, "w") as f:
        json.dump({"model": a.model, "max_tokens": a.max_tokens, "levels": results}, f, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
import os, re, signal, urllib.request
os.chdir("/kaggle/working")
ENV_VARS = dict(os.environ, VLLM_USE_V1="0", TOKENIZERS_PARALLELISM="false")
RESULTS = {"environment": ENV, "vllm_install_spec": INSTALLED, "experiments": {}, "failures": [], "skipped": []}
EXP = RESULTS["experiments"]

def save():
    with open("results.json", "w") as f:
        json.dump(RESULTS, f, indent=2)

def elapsed_min():
    return (time.time() - T_START) / 60

def log_hints(text):
    w = re.findall(r"Loading model weights took ([\d.]+) ?GB", text)
    b = re.findall(r"# (?:GPU|CUDA) blocks: (\d+)", text)
    return {"weights_gib_from_log": float(w[-1]) if w else None,
            "gpu_blocks_from_log": int(b[-1]) if b else None}

def gpu_used_mib():
    q = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True).stdout.split()
    return [int(x) for x in q]

def wait_gpu_free(limit_s=90):
    t = time.time()
    while time.time() - t < limit_s:
        if max(gpu_used_mib()) < 600:
            return True
        time.sleep(2)
    print("WARNING: GPU memory not released:", gpu_used_mib())
    return False

def run_group(args, timeout, log_path):
    with open(log_path, "w") as logf:
        p = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, env=ENV_VARS, start_new_session=True)
        try:
            rc = p.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL); p.wait(); rc = "timeout"
    return rc, open(log_path).read()

def run_offline(tag, timeout=900, **kw):
    args = [VPY, "offline_bench.py", "--out", f"tmp_{tag}.json"]
    for k, v in kw.items():
        if v is True:
            args.append("--" + k.replace("_", "-"))
        elif v not in (False, None):
            args += ["--" + k.replace("_", "-"), str(v)]
    t = time.time()
    rc, log = run_group(args, timeout, f"log_{tag}.txt")
    wait_gpu_free()
    if rc != 0:
        RESULTS["failures"].append({"tag": tag, "rc": rc, "log_tail": log[-3000:]})
        print(f"[{tag}] FAILED rc={rc}\n", log[-3000:]); save()
        return None
    res = json.load(open(f"tmp_{tag}.json"))
    res.update(log_hints(log)); res["tag"] = tag
    res["wall_including_startup_s"] = round(time.time() - t, 1)
    print(f"[{tag}] gen tok/s={res['generated_tokens_per_s']}  elapsed={res['elapsed_seconds']}s  "
          f"weights={res['weights_gib_from_log']}GiB  quality={res.get('quality', {}).get('exact_match')}  "
          f"(t={elapsed_min():.1f} min)")
    return res

def serve_and_bench(tag, model, quant=None, tp=1, levels="1,4,16,32", start_timeout=900):
    cmd = [VPY, "-m", "vllm.entrypoints.openai.api_server", "--model", model,
           "--dtype", "half", "--port", "8000", "--max-model-len", "2048",
           "--gpu-memory-utilization", "0.85", "--max-num-seqs", "128",
           "--tensor-parallel-size", str(tp), "--disable-log-requests",
           "--disable-custom-all-reduce", "--seed", "0"]
    if quant:
        cmd += ["--quantization", quant]
    logf = open(f"server_{tag}.log", "w")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=ENV_VARS, start_new_session=True)
    out = None
    try:
        t, ready = time.time(), False
        while time.time() - t < start_timeout and proc.poll() is None:
            try:
                ready = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).status == 200
            except Exception:
                ready = False
            if ready:
                break
            time.sleep(3)
        if not ready:
            raise RuntimeError("server did not become healthy")
        startup = round(time.time() - t, 1)
        mem_loaded = gpu_used_mib()
        rc, log = run_group([VPY, "online_client.py", "--model", model, "--levels", levels,
                             "--out", f"tmp_online_{tag}.json"], 1200, f"client_{tag}.txt")
        print(log[-2500:])
        if rc != 0:
            raise RuntimeError(f"client rc={rc}")
        out = json.load(open(f"tmp_online_{tag}.json"))
        out.update({"tag": tag, "quantization": quant, "tensor_parallel_size": tp,
                    "server_startup_s": startup, "nvidia_smi_used_mib_while_serving": mem_loaded})
    except Exception as e:
        logf.flush()
        RESULTS["failures"].append({"tag": tag, "error": repr(e), "log_tail": open(f"server_{tag}.log").read()[-3000:]})
        print(f"[{tag}] FAILED: {e!r}\n", open(f"server_{tag}.log").read()[-3000:])
    finally:
        try:
            os.killpg(proc.pid, signal.SIGTERM); proc.wait(timeout=60)
        except Exception:
            os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        logf.close()
        out_hints = log_hints(open(f"server_{tag}.log").read())
        if out:
            out.update(out_hints)
        wait_gpu_free()
        save()
        print(f"[{tag}] done (t={elapsed_min():.1f} min)")
    return out

M05, M05A = "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct-AWQ"
M15, M15A = "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
save()

## Experiment A — continuous batching (offline)

`vllm.LLM` with `Qwen2.5-0.5B-Instruct` fp16, 128 prompts × 128 output tokens (32 prompts for `max_num_seqs=1`, since
tokens/s is a rate and the sequential case is slow). Only `max_num_seqs` changes.

In [ ]:
rows = []
for mns in [1, 8, 32, 128]:
    r = run_offline(f"batch_mns{mns}", model=M05, max_num_seqs=mns,
                    num_prompts=32 if mns == 1 else 128, quality=(mns == 128))
    if r:
        rows.append(r)
EXP["a_offline_batching"] = {"model": M05, "dtype": "half", "rows": rows}
save()

## Experiment B — online serving (OpenAI-compatible server, streaming)

One server (`max_num_seqs=128`), streaming clients via `asyncio` + `httpx` at concurrency 1, 4, 16, 32.

In [ ]:
EXP["b_online_fp16_0.5B"] = serve_and_bench("fp16_05B", M05)

## Experiment C — fp16 vs AWQ (4-bit weights)

Same base model, same workload. Offline throughput at `max_num_seqs` 1 (latency-bound, memory-bandwidth-bound decode)
and 128 (compute-heavier), weight memory from vLLM's load log, KV-cache block count at the same
`gpu_memory_utilization=0.85`, and the 50-prompt exact-match check. Repeated at 1.5B if time allows.

In [ ]:
quant = {"0.5B": {}, "1.5B": {}}
quant["0.5B"]["fp16_mns128"] = EXP["a_offline_batching"]["rows"][-1] if rows and rows[-1]["config"]["max_num_seqs"] == 128 else None
quant["0.5B"]["fp16_mns1"] = rows[0] if rows and rows[0]["config"]["max_num_seqs"] == 1 else None
quant["0.5B"]["awq_mns1"] = run_offline("awq05_mns1", model=M05A, quantization="awq", max_num_seqs=1, num_prompts=32)
quant["0.5B"]["awq_mns128"] = run_offline("awq05_mns128", model=M05A, quantization="awq", max_num_seqs=128, quality=True)
EXP["c_quantization_offline"] = quant
save()
EXP["c_online_awq_0.5B"] = serve_and_bench("awq_05B", M05A, quant="awq")

In [ ]:
if elapsed_min() < 32:
    quant["1.5B"]["fp16_mns1"] = run_offline("fp16_15_mns1", model=M15, max_num_seqs=1, num_prompts=32)
    quant["1.5B"]["fp16_mns128"] = run_offline("fp16_15_mns128", model=M15, max_num_seqs=128, quality=True)
    quant["1.5B"]["awq_mns1"] = run_offline("awq15_mns1", model=M15A, quantization="awq", max_num_seqs=1, num_prompts=32)
    quant["1.5B"]["awq_mns128"] = run_offline("awq15_mns128", model=M15A, quantization="awq", max_num_seqs=128, quality=True)
else:
    RESULTS["skipped"].append(f"1.5B fp16 vs AWQ (time budget, t={elapsed_min():.1f} min)")
save()

## Experiment D — tensor parallel 1 vs 2

Same 0.5B fp16 model sharded across both T4s (`tensor_parallel_size=2`, custom all-reduce disabled, so all-reduce goes
through NCCL over PCIe). Baselines are the TP=1 runs from A and B.

In [ ]:
if ENV.get("gpu_count", 0) >= 2:
    tp2_off = run_offline("tp2_mns128", model=M05, tp=2, max_num_seqs=128)
    tp2_off1 = run_offline("tp2_mns1", model=M05, tp=2, max_num_seqs=1, num_prompts=32)
    EXP["d_tensor_parallel"] = {
        "model": M05,
        "offline": {"tp1_mns128": quant["0.5B"]["fp16_mns128"], "tp2_mns128": tp2_off,
                    "tp1_mns1": quant["0.5B"]["fp16_mns1"], "tp2_mns1": tp2_off1},
    }
    save()
    EXP["d_tensor_parallel"]["online_tp2"] = serve_and_bench("tp2_05B", M05, tp=2, levels="1,16")
else:
    RESULTS["skipped"].append(f"tensor parallel: only {ENV.get('gpu_count')} GPU visible")
save()

## Experiment E — prefix caching with a shared long system prompt

128 requests share a long system prompt (30 rules, roughly 1k tokens; exact prompt token totals are in the table);
32 output tokens each so prefill dominates.
Warm-up uses an unrelated prompt, so the first request still computes the shared prefix in the "on" case.

In [ ]:
if elapsed_min() < 45:
    EXP["e_prefix_caching"] = {
        "off": run_offline("prefix_off", model=M05, shared_prefix=True, max_tokens=32),
        "on": run_offline("prefix_on", model=M05, shared_prefix=True, max_tokens=32, prefix_caching=True),
    }
else:
    RESULTS["skipped"].append(f"prefix caching (time budget, t={elapsed_min():.1f} min)")
RESULTS["skipped"].append("gpu_memory_utilization / KV-cache size sweep (not run: workload never exhausts KV cache for these model sizes)")
RESULTS["total_runtime_min"] = round(elapsed_min(), 1)
save()
print("total runtime (min):", RESULTS["total_runtime_min"])

## Results tables

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)

def off_row(name, r):
    if not r:
        return {"config": name, "status": "failed/skipped"}
    q = r.get("quality") or {}
    return {"config": name, "max_num_seqs": r["config"]["max_num_seqs"], "tp": r["config"]["tp"],
            "requests": r["num_requests"], "gen_tok_s": r["generated_tokens_per_s"],
            "total_tok_s": r["total_tokens_per_s"], "elapsed_s": r["elapsed_seconds"],
            "weights_GiB": r.get("weights_gib_from_log"), "kv_blocks": r.get("num_gpu_kv_blocks"),
            "exact_match": q.get("exact_match")}

print("A. Offline batching (0.5B fp16)")
print(pd.DataFrame([off_row(f"mns={r['config']['max_num_seqs']}", r) for r in EXP["a_offline_batching"]["rows"]]).to_string(index=False))

print("\nC. Quantization (offline)")
print(pd.DataFrame([off_row(f"{size} {k}", v) for size, d in EXP["c_quantization_offline"].items() for k, v in d.items()]).to_string(index=False))

def online_rows(name, o):
    if not o:
        return [{"server": name, "status": "failed/skipped"}]
    return [{"server": name, "conc": l["concurrency"], "reqs": l["num_requests"], "errors": l["errors"],
             "out_tok_s": l["output_tokens_per_s"], "ttft_p50": l["ttft_ms"]["p50"], "ttft_p95": l["ttft_ms"]["p95"],
             "itl_p50": l["itl_ms"]["p50"], "itl_p95": l["itl_ms"]["p95"], "tpot_p50": l["tpot_ms"]["p50"],
             "e2e_p50": l["e2e_ms"]["p50"], "e2e_p95": l["e2e_ms"]["p95"]} for l in o["levels"]]

servers = [("0.5B fp16 TP1", EXP.get("b_online_fp16_0.5B")), ("0.5B AWQ TP1", EXP.get("c_online_awq_0.5B")),
           ("0.5B fp16 TP2", (EXP.get("d_tensor_parallel") or {}).get("online_tp2"))]
print("\nB/C/D. Online serving (latencies in ms)")
print(pd.DataFrame([row for n, o in servers for row in online_rows(n, o)]).to_string(index=False))

if "d_tensor_parallel" in EXP:
    print("\nD. Tensor parallel (offline)")
    print(pd.DataFrame([off_row(k, v) for k, v in EXP["d_tensor_parallel"]["offline"].items()]).to_string(index=False))
if "e_prefix_caching" in EXP:
    print("\nE. Prefix caching (offline, shared system prompt, 32 output tokens)")
    print(pd.DataFrame([dict(off_row(k, v), prompt_tokens=(v or {}).get("prompt_tokens"), req_s=(v or {}).get("requests_per_s"))
                        for k, v in EXP["e_prefix_caching"].items()]).to_string(index=False))
print("\nFailures:", [f.get("tag") for f in RESULTS["failures"]])
print("Skipped:", RESULTS["skipped"])

## Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

gpu = ENV["gpus"][0]["name"] if ENV.get("gpus") else "GPU"

rows_a = EXP["a_offline_batching"]["rows"]
if rows_a:
    fig, ax = plt.subplots(figsize=(6, 4))
    xs = [r["config"]["max_num_seqs"] for r in rows_a]
    ax.plot(xs, [r["generated_tokens_per_s"] for r in rows_a], marker="o")
    ax.set_xscale("log", base=2); ax.set_xticks(xs); ax.set_xticklabels(xs)
    ax.set_xlabel("max_num_seqs"); ax.set_ylabel("generated tokens/s")
    ax.set_title(f"Continuous batching, Qwen2.5-0.5B fp16, 1x {gpu}"); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig("fig_batching.png", dpi=120); plt.close(fig)

srv = [(n, o) for n, o in servers if o]
if srv:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for n, o in srv:
        c = [l["concurrency"] for l in o["levels"]]
        axes[0].plot(c, [l["output_tokens_per_s"] for l in o["levels"]], marker="o", label=n)
        axes[1].plot(c, [l["ttft_ms"]["p50"] for l in o["levels"]], marker="o", label=n + " p50")
        axes[1].plot(c, [l["ttft_ms"]["p95"] for l in o["levels"]], marker="x", ls="--", label=n + " p95")
        axes[2].plot(c, [l["itl_ms"]["p50"] for l in o["levels"]], marker="o", label=n + " p50")
        axes[2].plot(c, [l["itl_ms"]["p95"] for l in o["levels"]], marker="x", ls="--", label=n + " p95")
    for ax, t in zip(axes, ["output tokens/s", "TTFT (ms)", "inter-token latency (ms)"]):
        ax.set_xscale("log", base=2); ax.set_xlabel("concurrency"); ax.set_title(t); ax.grid(alpha=.3); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig("fig_online.png", dpi=120); plt.close(fig)

labels, vals = [], []
for size, d in EXP["c_quantization_offline"].items():
    for k, v in d.items():
        if v:
            labels.append(f"{size}\n{k}"); vals.append(v["generated_tokens_per_s"])
if vals:
    fig, ax = plt.subplots(figsize=(max(6, len(vals) * 1.1), 4))
    ax.bar(range(len(vals)), vals, color=["tab:blue" if "fp16" in l else "tab:orange" for l in labels])
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("generated tokens/s"); ax.set_title("fp16 (blue) vs AWQ (orange), offline"); ax.grid(axis="y", alpha=.3)
    fig.tight_layout(); fig.savefig("fig_quantization.png", dpi=120); plt.close(fig)

for f in ["fig_batching.png", "fig_online.png", "fig_quantization.png"]:
    if os.path.exists(f):
        display(Image(f))

## What I take from this

The cell below derives the comparisons directly from `results.json` of this run (ratios, not hand-typed numbers),
so the text stays true if the run is repeated and numbers shift.

In [ ]:
def g(r, k="generated_tokens_per_s"):
    return r.get(k) if r else None

notes = []
a = {r["config"]["max_num_seqs"]: r for r in EXP["a_offline_batching"]["rows"]}
if 1 in a and 128 in a:
    notes.append(f"Batching: max_num_seqs=128 gives {g(a[128]) / g(a[1]):.1f}x the generated tokens/s of max_num_seqs=1 "
                 f"({g(a[1])} -> {g(a[128])} tok/s) on the same GPU and model.")
b = EXP.get("b_online_fp16_0.5B")
if b and len(b["levels"]) >= 2:
    lo, hi = b["levels"][0], b["levels"][-1]
    notes.append(f"Online fp16 0.5B: concurrency {lo['concurrency']} -> {hi['concurrency']} raises output tok/s "
                 f"{lo['output_tokens_per_s']} -> {hi['output_tokens_per_s']} while p50 ITL goes {lo['itl_ms']['p50']} -> "
                 f"{hi['itl_ms']['p50']} ms and p95 TTFT {lo['ttft_ms']['p95']} -> {hi['ttft_ms']['p95']} ms.")
for size, d in EXP["c_quantization_offline"].items():
    for m in ("mns1", "mns128"):
        f16, awq = d.get(f"fp16_{m}"), d.get(f"awq_{m}")
        if f16 and awq:
            notes.append(f"{size} {m}: AWQ/fp16 throughput ratio = {g(awq) / g(f16):.2f} "
                         f"({g(f16)} vs {g(awq)} tok/s).")
    f16, awq = d.get("fp16_mns128"), d.get("awq_mns128")
    if f16 and awq:
        notes.append(f"{size}: weights {f16.get('weights_gib_from_log')} GiB (fp16) vs {awq.get('weights_gib_from_log')} GiB (AWQ); "
                     f"KV blocks {f16.get('num_gpu_kv_blocks')} vs {awq.get('num_gpu_kv_blocks')}; "
                     f"exact match {f16.get('quality', {}).get('exact_match')} vs {awq.get('quality', {}).get('exact_match')} on 50 prompts.")
d = EXP.get("d_tensor_parallel")
if d:
    for m in ("mns128", "mns1"):
        t1, t2 = d["offline"].get(f"tp1_{m}"), d["offline"].get(f"tp2_{m}")
        if t1 and t2:
            notes.append(f"TP=2 vs TP=1 offline {m}: {g(t2) / g(t1):.2f}x ({g(t1)} vs {g(t2)} tok/s). "
                         + ("TP=2 is slower here: the model is too small for the PCIe all-reduce cost to pay off." if g(t2) < g(t1)
                            else "TP=2 is faster here."))
e = EXP.get("e_prefix_caching")
if e and e.get("on") and e.get("off"):
    notes.append(f"Prefix caching: requests/s {e['off']['requests_per_s']} (off) -> {e['on']['requests_per_s']} (on), "
                 f"{e['on']['requests_per_s'] / e['off']['requests_per_s']:.2f}x, with a shared system prompt.")
RESULTS["derived_observations"] = notes
save()
print("\n".join("- " + n for n in notes))

## Limitations

- Two small models (0.5B, 1.5B) on T4s; conclusions about quantization and TP do not transfer to large models on
  NVLink GPUs, where the trade-offs are different.
- vLLM is pinned to an older release (V0 engine) because newer engines target compute capability ≥ 8.0; current
  vLLM performance on modern GPUs is not measured here.
- Single run per configuration, no repeats or confidence intervals; Kaggle GPUs are shared infrastructure and
  run-to-run variance of a few percent is expected.
- Synthetic prompts and fixed output length (`ignore_eos`) measure the engine, not real-traffic length distributions.
- The 50-prompt exact-match check only detects gross quality loss; it is not a quality benchmark.
- Client and server share the same VM, so client CPU overhead is included in the latency numbers.